In [ ]:
!pip install lancedb tmdbv3api serpapi python-dotenv pandas openpyxl

In [8]:
!pip install tmdbv3api

In [ ]:
!pip install dotenv openai pandas openpyxl requests


In [ ]:
!pip uninstall agno -y
!pip install "agno[openai,sqlite,pdf]==2.5.5"

In [19]:
from agno.agent import Agent, RunOutput
#from agno.models.anthropic import Claude
from agno.db.sqlite import SqliteDb
from agno.models.message import Message
from agno.run import RunContext
from agno.memory import MemoryManager
from agno.tools.serper import SerperTools

from agno.models.azure import AzureOpenAI

import os
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  


In [29]:
#---------- Additional Provided  
import os
from dotenv import load_dotenv
import pandas as pd

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb
from agno.tools import tool

from agno.knowledge import Knowledge
from agno.knowledge.reader.pdf_reader import PDFReader

from agno.vectordb.lancedb import LanceDb
from agno.knowledge.embedder.openai import OpenAIEmbedder

from tmdbv3api import TMDb, Movie



In [30]:
# =====================================================
# DATABASE (Agno 2.5.5 Persistence Layer)
# =====================================================

agent_db = SqliteDb(db_file="agents.db")

In [31]:
load_dotenv()
#os.environ["ANTHROPIC_API_KEY"] = getenv("ANTHROPIC_API_KEY")
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [32]:
#model = Claude(id="claude-sonnet-4-6")
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [ ]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


In [ ]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
AGENT_SYSTEM_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)


In [34]:
# ---------------------------------------------------
# Load Data
# ---------------------------------------------------

abt_df = pd.read_excel(ABT_FILE)
taxonomy_df = pd.read_excel(TAXONOMY_FILE)

In [35]:
# =====================================================
# TMDB SETUP
# =====================================================

tmdb = TMDb()
tmdb.api_key = TMDB_API_KEY
movie_api = Movie()



In [37]:
# =====================================================
# KNOWLEDGE (AUTO RAG INDEXING)
# =====================================================

print("🔄 Initializing RAG knowledge base...")

pdf_reader = PDFReader(path=OSCARS_PDF)

embedder = OpenAIEmbedder(id="text-embedding-3-small")

vector_db = LanceDb(
    uri="./oscars_lancedb",
    table_name="oscars_embeddings"
)

oscars_knowledge = Knowledge(
    name="oscars_knowledge",
    readers={"oscars_pdf": pdf_reader},
    vector_db=vector_db,
    contents_db=agent_db,
    max_results=5
)

print("✅ RAG ready.")


🔄 Initializing RAG knowledge base...


INFO Embedder not provided, using OpenAIEmbedder as default.

✅ RAG ready.


In [38]:
# =====================================================
# TOOLS
# =====================================================

@tool
def search_taxonomy(query: str):
    """Match user genre preference to curated taxonomy clusters."""

    df = taxonomy_df.copy()

    if "custom_genre" not in df.columns:
        return {"error": "custom_genre column not found in taxonomy file."}

    matches = df[
        df["custom_genre"].str.contains(query, case=False, na=False)
    ]

    if matches.empty:
        return {"message": "No matching genre cluster found."}

    return matches.head(5).to_dict(orient="records")


@tool
def search_abt(genre: str):
    """
    Search enriched ABT dataset by genre.
    Rank by vote_average, then popularity.
    """

    df = abt_df.copy()

    # Ensure genre column exists
    if "genres" not in df.columns:
        return {"error": "Column 'genres' not found in ABT dataset."}

    # Filter by genre
    matches = df[df["genres"].str.contains(genre, case=False, na=False)]

    if matches.empty:
        return {"message": f"No movies found for genre: {genre}"}

    # Rank by vote_average then popularity
    matches = matches.sort_values(
        by=["vote_average", "popularity"],
        ascending=False
    )

    # Select structured output fields
    output_columns = [
        "title",
        "overview",
        "tagline",
        "genres",
        "vote_average",
        "vote_count",
        "popularity",
        "runtime",
        "original_language",
        "poster_path",
        "homepage",
        "budget",
        "revenue",
        "release_date",
        "tmdbId"
    ]

    existing_cols = [col for col in output_columns if col in matches.columns]

    return matches[existing_cols].head(5).to_dict(orient="records")


@tool
def search_tmdb_by_id(tmdb_id: int):
    return movie_api.details(tmdb_id)


serper_tool = SerperTools(api_key=SERPER_API_KEY)

In [39]:
# =====================================================
# SPECIALIST AGENTS
# =====================================================

recommender_agent = Agent(
    name="RecommenderAgent",
    model=model,
    tools=[search_taxonomy, search_abt, search_tmdb_by_id],
    db=agent_db,
    description="Handles personalized movie recommendations."
)

awards_agent = Agent(
    name="AwardsAgent",
    model=model,
    knowledge=oscars_knowledge,
    db=agent_db,
    description="Handles Oscar and award-related questions."
)

web_agent = Agent(
    name="WebAgent",
    model=model,
    tools=[serper_tool],
    db=agent_db,
    description="Handles latest releases and streaming queries."
)


# =====================================================
# ROUTER AGENT
# =====================================================

router_agent = Agent(
    name="RouterAgent",
    model=model,
    db=agent_db,
    description="""
Classify the user query into one of:
RECOMMENDER
AWARDS
WEB

Respond with ONLY one word.
"""
)


# =====================================================
# ROUTING FUNCTION
# =====================================================

def route_query(user_input):

    intent_response = router_agent.run(input=user_input)
    intent = intent_response.content.strip().upper()

    print(f"[DEBUG] Router selected: {intent}")

    if "AWARD" in intent:
        return awards_agent.run(input=user_input)

    elif "WEB" in intent:
        return web_agent.run(input=user_input)

    else:
        return recommender_agent.run(input=user_input)

In [40]:
if __name__ == "__main__":

    test_queries = [
        "Recommend a romantic comedy for date night.",
        "Who won Best Picture at the 2026 Oscars?",
        "What are the latest Bollywood movies in 2025?",
        "Is Oppenheimer available on Netflix?"
    ]

    for q in test_queries:
        print(f"\nUser: {q}")
        response = route_query(q)
        print("\nBot:")
        print(response.content)
        print("-" * 80)


User: Recommend a romantic comedy for date night.
[DEBUG] Router selected: RECOMMENDER

Bot:
I couldn't find the exact genre cluster for "romantic comedy." Would you like a recommendation from the genres romance or comedy instead? Or perhaps a combination of movies that are romantic and funny? Let me know your preference!
--------------------------------------------------------------------------------

User: Who won Best Picture at the 2026 Oscars?
[DEBUG] Router selected: AWARDS


INFO Found 0 documents


Bot:
I couldn't find the information about the Best Picture winner at the 2026 Oscars. The event might not have occurred yet or the results are not available in the knowledge base. Is there anything else you would like to inquire about?
--------------------------------------------------------------------------------

User: What are the latest Bollywood movies in 2025?
[DEBUG] Router selected: WEB

Bot:
The latest Bollywood movies in 2025 include:

1. Dhurandhar
2. Chhaava
3. Saiyaara
4. Mahavatar Narsimha
5. Homebound
6. Sitaare Zameen Par
7. Santosh
8. Superboys of Malegaon
9. The Mehta Boys
10. Stolen
11. Tanvi
12. 120 Bahadur (release date 21 November 2025)
13. Jugnuma
14. Haq
15. Crazxy
16. Anuja

These movies cover a range of genres including action, drama, and history. For more details, you can check sources like IMDb, Wikipedia, and Times of India lists. Would you like information about any specific movie?
------------------------------------------------------------------------

In [ ]:
# -------------------------------------------------------
# ---------------- RUN LOOP -----------------------------
# -------------------------------------------------------

# ======
# ==============================================
# RUN LOOP
# =====================================================

if __name__ == "__main__":

    print("🎬 Multi-Agent Movie Bot (Agno 2.5.5 — Production Stable)")
    print("Type 'exit' to quit.\n")

    session_id = "user_session_1"

    while True:
        user_input = input("User: ")

        if user_input.lower() == "exit":
            break

        response = route_query(user_input)

        print("\nBot:\n")
        print(response.content)
        print("-" * 80)

🎬 Multi-Agent Movie Bot (Agno 2.5.5 — Production Stable)
Type 'exit' to quit.

[DEBUG] Router selected: WEB

Bot:

Hello! How can I assist you today?
--------------------------------------------------------------------------------


In [ ]:
"""###"""🚀 Example Queries to Test

"Recommend a romantic comedy for date night."

"Who won Best Picture at the 2026 Oscars?"

"What are the latest Bollywood movies in 2025?"

"Is Oppenheimer available on Netflix?""""###"""

In [7]:
memory_manager = MemoryManager(
    model=model,
    db=agent_db,
    additional_instructions="""
    Capture the user's movie preferences, including who they are watching with and their movie genre, language preferences.
    """,
)

In [9]:
agent = Agent(
    model=model,
    tools=[SerperTools()],
    session_state={"preferences": []},
    db=SqliteDb(db_file="tmp/agents.db"),
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent state (preferences) is: {preferences}",
    markdown=True,
    debug_mode=False,
    memory_manager=memory_manager,
    enable_agentic_memory=True,
    add_datetime_to_context=True,
    add_history_to_context=True,
    num_history_runs=5
 #   show_tool_calls           = True,   # debug: show which tools fire
)


print("Movie Agent is running. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)

    # ✅ Print session state after each run
    print("\nSession State:")
    print(run.session_state)

    print("\n" + "-"*50 + "\n")
#run = agent.run("What is the story for the movie Toy Story in 20 words or less", stream=False)

Movie Agent is running. Type 'exit' to quit.

Your input:  I would like to watch the latest bollywood movie with my sweetheart

Agent:
Great! Watching a Bollywood movie with your sweetheart sounds lovely. 

To narrow down the perfect latest Bollywood movie for you, could you tell me what kind of genre or mood you're in? For example, are you leaning towards romance, drama, comedy, action, or something else?

Session State:
{'preferences': []}

--------------------------------------------------

Your input:  I lile romantic comedy

Agent:
Romantic comedy is a fantastic mood for a date movie! 

Just to be sure, is there any particular preference for when the movie was released? Since you mentioned "latest," are you interested in Bollywood romantic comedies from the past year or is a slightly older movie okay too?

Session State:
{'preferences': []}

--------------------------------------------------

Your input:  I am only interested in movies released in 2025

Agent:
For a romantic comed

In [18]:
print(run.content)

Toy Story: A cowboy doll feels threatened when a new spaceman toy arrives, leading to adventures and friendship.
